In [1]:
### Importing the required library 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import MolStandardize
import joblib

from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors,rdMolDescriptors


In [2]:
### Installation of the basic library 
from rdkit import Chem,DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import Crippen
import numpy as np
import pandas as pd

In [3]:
### Reading the preprocess data from disk 
train_set=pd.read_csv('../data/final_data/final_unique_train.csv')

test_set=pd.read_csv('../data/final_data/final_unique_test.csv')

print(train_set.shape)
print(test_set.shape)

(17937, 8)
(1282, 8)


In [4]:
train_smiles_list=train_set[['smiles_canon']]
test_smiles_list=test_set[['smiles_canon']]

In [5]:

### Importing the function of the utility file to generate the descriptors and others evaluation matrics...  
import utilities

2025-04-19 08:55:52.351414: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, MolSurf, rdMolDescriptors
import pandas as pd
from rdkit import Chem

In [7]:
### Generate 4 descriptors ....
df4_train=utilities.generate4(train_set.smiles_canon)
df4_test=utilities.generate4(test_set.smiles_canon)
### Generate 17 descriptors ....
df17_train=utilities.generate17(train_set.smiles_canon)
df17_test=utilities.generate17(test_set.smiles_canon)
### Generate 123 descriptors ....
df123_train=utilities.generate123(train_set.smiles_canon)
df123_test=utilities.generate123(test_set.smiles_canon)
### Generate 38 feature engineered based on the structure of the smiles ....
df38_train=utilities.generate_features38(train_set.smiles_canon)
df38_test=utilities.generate_features38(test_set.smiles_canon)
### Generate 7 funnctional groups
df7_train=utilities.get_functional_groups(train_set.smiles_canon)
df7_test=utilities.get_functional_groups(test_set.smiles_canon)
### Fingerprint 128....
df128_train=utilities.fingerprint(train_set.smiles_canon,2,128)
df128_test=utilities.fingerprint(test_set.smiles_canon,2,128)
### Fingerprint 256....
df256_train=utilities.fingerprint(train_set.smiles_canon,2,256)
df256_test=utilities.fingerprint(test_set.smiles_canon,2,256)
### Fingerprint 512....
df512_train=utilities.fingerprint(train_set.smiles_canon,2,512)
df512_test=utilities.fingerprint(test_set.smiles_canon,2,512)
### Fingerprint 1024....
df1024_train=utilities.fingerprint(train_set.smiles_canon,2,1024)
df1024_test=utilities.fingerprint(test_set.smiles_canon,2,1024)

[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:13] WARNING: not removing hydrogen atom without neighbors
[08:56:24] WARNING: not removing hydrogen atom without neighbors
[08:56:24] WARNING: not removing hydrogen atom without neighbors
[08:56:24] WARNING: not removing hydrogen atom without neighbors
[08:56:24] WARNING: not r

In [8]:
### Taking out the Solubility from the dataset 
y_train=train_set['LogS']
y_test=test_set['LogS']

In [9]:
### Importing the library 
import xgboost #as xgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_squared_error
import numpy as np

In [10]:
### for the fisrt 4 features 
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []

# Predictions storage for test set
test_predictions = []

# Loop through each fold
for train_index, val_index in kf.split(df4_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df4_train.iloc[train_index], df4_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    
 
    
    # Test predictions for this fold
    y_test_pred = model.predict(df4_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")




Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.5588, Std = 0.0024
RMSE: Mean = 0.7200, Std = 0.0051
R²: Mean = 0.8757, Std = 0.0018


In [11]:
### For the 17 features 
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df17_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df17_train.iloc[train_index], df17_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df17_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4591, Std = 0.0020
RMSE: Mean = 0.6022, Std = 0.0032
R²: Mean = 0.9131, Std = 0.0009


In [12]:
### for the 125 features 
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df123_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df123_train.iloc[train_index], df123_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df123_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4184, Std = 0.0012
RMSE: Mean = 0.5671, Std = 0.0011
R²: Mean = 0.9229, Std = 0.0003


In [13]:
df253_train = pd.concat([df123_train, df128_train], axis=1)
df253_test = pd.concat([df123_test, df128_test], axis=1)

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df253_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df253_train.iloc[train_index], df253_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df253_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4188, Std = 0.0014
RMSE: Mean = 0.5671, Std = 0.0026
R²: Mean = 0.9229, Std = 0.0007


In [15]:
df260_train = pd.concat([df253_train, df7_train], axis=1)
df260_test = pd.concat([df253_test, df7_test], axis=1)

In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df260_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df260_train.iloc[train_index], df260_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df260_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



In [ ]:
df298_train = pd.concat([df260_train, df38_train], axis=1)
df298_test = pd.concat([df260_test, df38_test], axis=1)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df298_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df298_train.iloc[train_index], df298_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df298_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4148, Std = 0.0027
RMSE: Mean = 0.5632, Std = 0.0043
R²: Mean = 0.9239, Std = 0.0012


In [25]:
df163_train = pd.concat([df123_train, df38_train], axis=1)
df163_test = pd.concat([df123_test, df38_test], axis=1)

In [26]:
df170_train = pd.concat([df163_train, df7_train], axis=1)
df170_test = pd.concat([df163_test, df7_test], axis=1)

In [27]:
df682_train = pd.concat([df170_train, df512_train], axis=1)
df682_test = pd.concat([df170_test, df512_test], axis=1)

In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df682_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df682_train.iloc[train_index], df682_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df682_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4164, Std = 0.0028
RMSE: Mean = 0.5629, Std = 0.0033
R²: Mean = 0.9240, Std = 0.0009


In [29]:
df1194_train = pd.concat([df170_train, df1024_train], axis=1)
df1194_test = pd.concat([df170_test, df1024_test], axis=1)

In [30]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Metrics list
mae_scores = []
rmse_scores = []
r2_scores = []



# Loop through each fold
for train_index, val_index in kf.split(df1194_train):
    # Splitting train data into training and validation for the current fold
    X_train_fold, X_val_fold = df1194_train.iloc[train_index], df1194_train.iloc[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
    

    
    # Initialize model (you can replace this with any model)
    model = xgboost_model = xgboost.XGBRegressor(learning_rate =0.01,n_estimators=2000,max_depth=8,min_child_weight=4,gamma=0,subsample=0.7,
                                           colsample_bytree=0.8,nthread=2,scale_pos_weight=1,seed=27)
    model.fit(X_train_fold, y_train_fold)
    

    
    
    
    # Test predictions for this fold
    y_test_pred = model.predict(df1194_test)
    test_predictions.append(y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2 = r2_score(y_test, y_test_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)


# Calculate mean and standard deviation of metrics across all folds
mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)



# Output results
print("Mean and Standard Deviation of Metrics Across 5 Folds:")
print(f"MAE: Mean = {mean_mae:.4f}, Std = {std_mae:.4f}")
print(f"RMSE: Mean = {mean_rmse:.4f}, Std = {std_rmse:.4f}")
print(f"R²: Mean = {mean_r2:.4f}, Std = {std_r2:.4f}")



Mean and Standard Deviation of Metrics Across 5 Folds:
MAE: Mean = 0.4227, Std = 0.0034
RMSE: Mean = 0.5671, Std = 0.0053
R²: Mean = 0.9229, Std = 0.0014


In [ ]:
### End here ...